# UrbanPulse AI - Smart Bin Fill-Level Prediction
## Notebook 01: Dataset Inspection & ML Feasibility Analysis

**Role:** Senior Machine Learning Engineer  
**Task:** Raw Dataset Inspection & Validation  
**Module:** Smart City Waste Management - Bin Fill-Level Prediction  

---

### Objectives:
1. Inspect existing project structure.
2. Locate and inspect the raw ZIP archive in `datasets/raw/` non-destructively.
3. Identify actual file format and available assets.
4. Extract and inspect data using Pandas.
5. Profile dataset dimensions, schema, summary statistics, missing values, duplicates, and target variable distribution.
6. Evaluate whether the dataset is genuinely suitable for a supervised machine learning prediction task.

### Step 1: Inspect Project Directory Structure

In [1]:
import os
from pathlib import Path

# Establish project root
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parents[1] if cwd.name == 'notebooks' else cwd
print(f'Project root directory: {PROJECT_ROOT}')

# Display top-level repository structure
print('\n--- Top-Level Directory Structure ---')
for item in sorted(PROJECT_ROOT.iterdir()):
    if item.name.startswith('.'):
        continue
    item_type = '[DIR] ' if item.is_dir() else '[FILE]'
    print(f'{item_type} {item.name}')

Project root directory: C:\Users\shiva\UrbanPulse-AI

--- Top-Level Directory Structure ---
[DIR]  backend
[DIR]  datasets
[DIR]  docs
[DIR]  frontend
[DIR]  ml
[DIR]  tests


### Step 2: Locate ZIP Dataset inside datasets/raw/ (Non-Destructive)

In [2]:
raw_dir = PROJECT_ROOT / 'datasets' / 'raw'
zip_files = list(raw_dir.glob('*.zip'))

print(f'Raw dataset directory: {raw_dir}')
print(f'ZIP archives found: {len(zip_files)}')
for zf in zip_files:
    size_kb = zf.stat().st_size / 1024
    print(f' - Archive: {zf.name} ({size_kb:.2f} KB, {zf.stat().st_size:,} bytes)')

Raw dataset directory: C:\Users\shiva\UrbanPulse-AI\datasets\raw
ZIP archives found: 1
 - Archive: Smart Bin Insights Data Set for Unveiling Waste Management Trends.zip (258.17 KB, 264,369 bytes)


### Step 3: Identify ZIP Contents without Modifying or Deleting Original Archive

In [3]:
import zipfile

zip_path = zip_files[0]
with zipfile.ZipFile(zip_path, 'r') as z:
    print(f'Inspecting contents of: {zip_path.name}')
    print(f'Total items inside archive: {len(z.infolist())}')
    print('-' * 60)
    for info in z.infolist():
        print(f'Filename:          {info.filename}')
        print(f'Compressed Size:   {info.compress_size:,} bytes')
        print(f'Uncompressed Size: {info.file_size:,} bytes')

Inspecting contents of: Smart Bin Insights Data Set for Unveiling Waste Management Trends.zip
Total items inside archive: 1
------------------------------------------------------------
Filename:          Smart Bin Insights Data Set for Unveiling Waste Management Trends/Smart Bin.docx
Compressed Size:   264,095 bytes
Uncompressed Size: 281,367 bytes


### Step 4: Determine Actual Dataset Format & Extract

> **Key Finding**: The archive does NOT contain standard machine learning tabular formats (such as `.csv`, `.tsv`, `.parquet`, or `.json`).
> Instead, it contains a single Microsoft Word document (`.docx`).

We extract the archive into `datasets/raw/` preserving the original `.zip` completely.

In [4]:
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(raw_dir)

extracted_files = [f for f in raw_dir.rglob('*') if f.is_file() and f.suffix != '.zip']
print('Extracted files in datasets/raw/:')
for f in extracted_files:
    print(f' - {f.relative_to(raw_dir)} ({f.stat().st_size:,} bytes)')

Extracted files in datasets/raw/:
 - Smart Bin Insights Data Set for Unveiling Waste Management Trends\Smart Bin.docx (281,367 bytes)


### Step 5: Attempting Direct Pandas Inspection
Let us test standard Pandas tabular loaders (`pd.read_csv` and `pd.read_excel`) on the extracted file to demonstrate compatibility.

In [5]:
import pandas as pd

target_docx = extracted_files[0]
print(f'Inspecting file with Pandas: {target_docx.name}\n')

# Attempt 1: pd.read_csv
try:
    pd.read_csv(target_docx)
    print('pd.read_csv succeeded.')
except Exception as e:
    print(f'pd.read_csv failed (expected): {type(e).__name__} - {e}')

# Attempt 2: pd.read_excel
try:
    pd.read_excel(target_docx)
    print('pd.read_excel succeeded.')
except Exception as e:
    print(f'pd.read_excel failed (expected): {type(e).__name__} - {e}')

Inspecting file with Pandas: Smart Bin.docx

pd.read_csv failed (expected): UnicodeDecodeError - 'utf-8' codec can't decode byte 0xe6 in position 15: invalid continuation byte
pd.read_excel failed (expected): OptionError - No such keys(s): 'io.excel.zip.reader'


### Step 6: Deep Document Inspection & Sensor Telemetry Extraction

Forensic inspection of `Smart Bin.docx` reveals that it is an academic paper titled *"Smart Bin Insights: Data Set for Unveiling Waste Management Trends"* by researchers from Mumbai engineering colleges.

Inside the document package (`word/embeddings/oleObject2.bin`), an embedded OriginLab OLE compound binary object stores the underlying numerical data used in the paper plots:
- `Book1_A`: Timestamp (Julian day numbers, 2458295.0 to 2459030.0)
- `Book1_B`: Latest bin fullness level (`latestFullness`, discrete scale 0 to 10)

We extract this binary series directly into a Pandas DataFrame.

In [6]:
import struct

# Inspect internal structure of DOCX package
with zipfile.ZipFile(target_docx, 'r') as docx_zip:
    ole_data = docx_zip.read('word/embeddings/oleObject2.bin')

# Extract Book1_A (Julian timestamp) - 683 data points
pos_a = ole_data.find(b'Book1_A\x00')
start_a = ole_data.find(bytes.fromhex('00 00 00 80 5b c1 42 41'), pos_a)
vals_a = [struct.unpack('<d', ole_data[start_a + i*10 : start_a + i*10 + 8])[0] for i in range(683)]

# Extract Book1_B (latestFullness) - 683 data points
pos_b = ole_data.find(b'Book1_B\x00')
start_b = ole_data.find(bytes.fromhex('00 00 00 00 00 00 18 40'), pos_b)
vals_b = [struct.unpack('<d', ole_data[start_b + i*10 : start_b + i*10 + 8])[0] for i in range(683)]

# Construct Pandas DataFrame
df = pd.DataFrame({
    'timestamp_julian': vals_a,
    'latestFullness': vals_b
})

# Convert Julian day number to Gregorian datetime
df['datetime'] = pd.to_datetime(df['timestamp_julian'], unit='D', origin='julian')

print(f'Extracted DataFrame with {len(df)} records and {df.shape[1]} columns.')

Extracted DataFrame with 683 records and 3 columns.


### Step 7: Complete Profiling of Extracted Dataset with Pandas

We inspect the required metrics:
1. Dataset dimensions
2. Column names and data types
3. First five records
4. Missing values
5. Duplicate records
6. Unique values in categorical / discrete columns
7. Summary statistics

In [7]:
import numpy as np

print('=' * 60)
print('1. DATASET DIMENSIONS')
print('=' * 60)
print(f'Total Rows:    {df.shape[0]}')
print(f'Total Columns: {df.shape[1]}')
print(f'Shape:         {df.shape}')

print('\n' + '=' * 60)
print('2. COLUMN NAMES & DATA TYPES')
print('=' * 60)
print(df.dtypes)

print('\n' + '=' * 60)
print('3. FIRST FIVE RECORDS')
print('=' * 60)
print(df.head(5).to_string())

print('\n' + '=' * 60)
print('4. LAST FIVE RECORDS')
print('=' * 60)
print(df.tail(5).to_string())

print('\n' + '=' * 60)
print('5. MISSING VALUES CHECK')
print('=' * 60)
print(df.isnull().sum())

print('\n' + '=' * 60)
print('6. DUPLICATE RECORDS')
print('=' * 60)
total_dups = df.duplicated().sum()
ts_dups = df['timestamp_julian'].duplicated().sum()
print(f'Exact duplicates across all columns: {total_dups}')
print(f'Duplicate timestamps:                {ts_dups}')

print('\n' + '=' * 60)
print('7. UNIQUE VALUES IN TARGET COLUMN (latestFullness)')
print('=' * 60)
unique_vals = np.sort(df['latestFullness'].unique())
print(f'Unique levels: {unique_vals}')
print('\nValue Counts / Frequency Distribution:')
print(df['latestFullness'].value_counts().sort_index().to_string())

print('\n' + '=' * 60)
print('8. DESCRIPTIVE SUMMARY STATISTICS')
print('=' * 60)
print(df.describe(include='all'))

1. DATASET DIMENSIONS
Total Rows:    683
Total Columns: 3
Shape:         (683, 3)

2. COLUMN NAMES & DATA TYPES
timestamp_julian           float64
latestFullness             float64
datetime            datetime64[ns]
dtype: object

3. FIRST FIVE RECORDS
   timestamp_julian  latestFullness            datetime
0         2458295.0             6.0 2018-06-25 12:00:00
1         2458296.0             0.0 2018-06-26 12:00:00
2         2458297.0             2.0 2018-06-27 12:00:00
3         2458298.0             8.0 2018-06-28 12:00:00
4         2458299.0             6.0 2018-06-29 12:00:00

4. LAST FIVE RECORDS
     timestamp_julian  latestFullness            datetime
678         2459026.0             0.0 2020-06-25 12:00:00
679         2459027.0             4.0 2020-06-26 12:00:00
680         2459028.0             6.0 2020-06-27 12:00:00
681         2459029.0             0.0 2020-06-28 12:00:00
682         2459030.0             0.0 2020-06-29 12:00:00

5. MISSING VALUES CHECK
timestamp_julia

### Step 8: Machine Learning Feasibility Analysis

#### 1. Target Variable Formulation:
- **Variable**: `latestFullness`
- **Regression formulation**: Continuous fill level target (scale 0.0 to 10.0 or normalized 0% - 100%).
- **Classification formulation**: Ordinal/Multiclass target with 6 discrete levels: `[0.0, 2.0, 4.0, 6.0, 8.0, 10.0]`, or discretized into status classes (`Empty [0]`, `Low [2]`, `Medium [4]`, `High [6]`, `Critical [8-10]`).

#### 2. Suitability Assessment for Supervised ML:
| Assessment Criterion | Current Dataset Status | Production / Genuine ML Requirement | Feasibility Verdict |
| :--- | :--- | :--- | :--- |
| **File Format** | Microsoft Word Document (`.docx`) | Tabular data (`.csv`, `.parquet`, database) | ❌ Incompatible |
| **Feature Dimensionality** | 1 feature (`timestamp`) | Multi-feature telemetry (fill %, distance, weight, battery, temperature) | ❌ Severely Deficient |
| **Spatial Coverage** | Single bin (unidentified) | Multi-bin network with `bin_id`, GPS coordinates, zone/route | ❌ Single Bin Only |
| **Sample Size** | 683 temporal readings (~2 years) | Thousands to millions of sensor readings across fleet | ❌ Inadequate |
| **Data Quality** | 24 duplicate timestamps/readings | Clean, deduplicated, regular time-series | ⚠️ Quality Issues |
| **Ground Truth Events** | No collection event logs | Collection timestamps, emptyings, maintenance flags | ❌ Absent |

#### Conclusion & Recommendation:
- **Verdict**: The provided ZIP file is **not genuinely suitable** for training a robust supervised machine learning model for the UrbanPulse AI platform.
- **Recommendation**: Source a legitimate, multi-sensor municipal IoT waste management dataset (such as the Melbourne Open Data Smart Bins sensor readings or equivalent benchmark dataset) containing rich sensor features (`bin_id`, `latitude`, `longitude`, `sensor_distance`, `fill_level_percentage`, `battery_level`, `timestamp`, `waste_type`) to build the fill-level prediction engine.